In [1]:
import numpy as np
import pandas as pd
import torch
import json
from collections import defaultdict
from sklearn.preprocessing import StandardScaler

In [2]:
import json
with open(r"C:\Users\ps302\OneDrive\Desktop\Recommend\src\data\processed\vocab_dict_data\categorical_maps.json", "rb") as file:
    categorical_maps = json.load(file)
categorical_maps

{'department': {'Agricultural Engineering': 1,
  'Agricultural Sciences': 2,
  'Agricultural and Biological Engineering': 3,
  'Agricultural and Environmental Sciences': 4,
  'Agronomy': 5,
  'Amity Institute of Applied Sciences (Noida)': 6,
  'Amity School of Physical Sciences': 7,
  'Amrut Mody School of Management': 8,
  'Biochemistry': 9,
  'Bioinformatics': 10,
  'Bioinformatics and Computational Biology': 11,
  'Biological Engineering': 12,
  'Biological Sciences': 13,
  'Chemical Engineering': 14,
  'Chemical and Biochemical Processing Division': 15,
  'Chemistry': 16,
  'Civil and Environmental Engineering': 17,
  'Computational Biology': 18,
  'Computer Science': 19,
  'Computer Science and Artificial Intelligence Laboratory': 20,
  'Computer Science and Artificial Intelligence Laboratory (CSAIL)': 21,
  'Computer Science and Engineering': 22,
  'Computer Science and Media Arts': 23,
  'Computer Science and Media Lab': 24,
  'Computer Science and Technology': 25,
  'Department

In [3]:
PHD_prefinal_dataset = defaultdict()

In [4]:
df = pd.read_json(r"C:\Users\ps302\OneDrive\Desktop\Recommend\src\data\processed\phd\sop\gpt_extracted_data.json")
df.head()

,name,research_interests,expertise,department,university,country,publications,publication_count,citation_count,years_experience,sop_paragraphs,category,sub_domain,scholar_id,source_file,sop_paragraphs_json,full_sop_markdown,word_count
0,Maya L. Patel,"[Autonomous navigation for aerial robotics, Mu...","[Python, C++, Robot Operating System (ROS), Te...",Electrical and Computer Engineering,Stanford University,United States,[Learning Safe Maneuvers for Quadrotor Swarms ...,2,34,3,[From watching a flock of drones weave through...,Engineering & Technology,AI & Autonomous Systems,150,generated_sop_150.pdf,"[""From watching a flock of drones weave throug...",From watching a flock of drones weave through ...,430
1,Alejandro R. Gómez,[Satellite remote sensing for climate change m...,"[Python, R, MATLAB, Google Earth Engine, ENVI,...",Civil and Environmental Engineering,University of Cambridge,United Kingdom,[Deep Learning‑Based Atmospheric Correction fo...,2,21,4,[When I first examined a series of Sentinel‑2 ...,Engineering & Technology,AI & Autonomous Systems,151,generated_sop_151.pdf,"[""When I first examined a series of Sentinel\u...",When I first examined a series of Sentinel‑2 i...,445
2,Aisha Khan,"[Autonomous navigation for aerial robotics, Mu...","[Python, C++, Robot Operating System (ROS), Te...",Electrical and Computer Engineering,Massachusetts Institute of Technology,United States,[Learning Safe Maneuvers for Quadrotor Swarms ...,2,27,3,[From the moment I programmed my first line‑of...,Engineering & Technology,AI & Autonomous Systems,152,generated_sop_152.pdf,"[""From the moment I programmed my first line\u...",From the moment I programmed my first line‑of‑...,453
3,Mateo Alvarez,[Satellite remote sensing for environmental mo...,"[Python (NumPy, Pandas, Scikit‑learn), R (sp, ...",Civil and Environmental Engineering,University of Cambridge,United Kingdom,[Integrating SAR and Optical Data for Flood Ma...,3,45,4,"[Growing up along the Pacific coast of Chile, ...",Engineering & Technology,AI & Autonomous Systems,153,generated_sop_153.pdf,"[""Growing up along the Pacific coast of Chile,...","Growing up along the Pacific coast of Chile, I...",443
4,Aria L. Chen,[Robust perception for autonomous aerial vehic...,"[Python, C++, ROS2, TensorFlow, PyTorch, JAX, ...",Electrical Engineering and Computer Science,Massachusetts Institute of Technology,United States,[Safe Multi‑Agent Reinforcement Learning for U...,3,27,4,[From the moment I piloted a quadcopter throug...,Engineering & Technology,AI & Autonomous Systems,154,generated_sop_154.pdf,"[""From the moment I piloted a quadcopter throu...",From the moment I piloted a quadcopter through...,479


In [5]:
# Categorcial Data -> 
categorical_col = ["department", "university", "country", "sub_domain", "category"]
# df1 = df[["department", "university", "country", "sub_domain", "category"]]
# df2 = prof_df[["department"]]
# df2.head()

In [6]:
import pandas as pd
import torch

def preprocess_categorical_dataset(df, categorical_cols, categorical_maps, id_col="scholar_id", device=None):
    """
    Maps categorical columns to IDs, converts each row's categorical IDs to a PyTorch tensor,
    and returns a list of dictionaries in the format: {id_col: val, 'cat_ids': tensor([...])}
    """
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        
    # 1. Map all categorical columns in a vectorized way
    mapped_df = pd.DataFrame(index=df.index)
    for col in categorical_cols:
        col_map = categorical_maps.get(col, {})
        # Map values to their IDs, defaulting to 0 (UNKNOWN's ID) if not found
        mapped_df[col] = (
            df[col]
            .fillna("UNKNOWN")
            .astype(str)
            .map(lambda x: col_map.get(x, col_map.get("UNKNOWN", 0)))
        )
    
    # 2. Convert mapped values to a 2D numpy array, then to a PyTorch tensor
    # Shape of all_cat_tensors: (num_rows, num_categorical_columns)
    all_cat_tensors = torch.tensor(mapped_df[categorical_cols].values, dtype=torch.long, device=device)
    
    # 3. Assemble the final list of dicts
    prefinal_dataset = []
    ids = df[id_col].values
    
    for idx, row_id in enumerate(ids):
        prefinal_dataset.append({
            id_col: int(row_id),
            "cat_ids": all_cat_tensors[idx] # Slice the tensor for the current row
        })
        
    return prefinal_dataset


In [7]:
# Select the device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Run the pre-processing function
PHD_prefinal_dataset = preprocess_categorical_dataset(
    df=df,
    categorical_cols=categorical_col,
    categorical_maps=categorical_maps,
    id_col="scholar_id",
    device=device
)

# Example output verification:
print("First item in the dataset:")
print(PHD_prefinal_dataset[0])


First item in the dataset:
{'scholar_id': 150, 'cat_ids': tensor([117,   8,   7,   1,   4])}


In [ ]:
# PHD_prefinal_dataset

{'scholar_id': 150, 'cat_ids': tensor([117,   8,   7,   1,   4])}

In [ ]:
# df.head()

,name,research_interests,expertise,department,university,country,publications,publication_count,citation_count,years_experience,sop_paragraphs,category,sub_domain,scholar_id,source_file,sop_paragraphs_json,full_sop_markdown,word_count
0,Maya L. Patel,"[Autonomous navigation for aerial robotics, Mu...","[Python, C++, Robot Operating System (ROS), Te...",Electrical and Computer Engineering,Stanford University,United States,[Learning Safe Maneuvers for Quadrotor Swarms ...,2,34,3,[From watching a flock of drones weave through...,Engineering & Technology,AI & Autonomous Systems,150,generated_sop_150.pdf,"[""From watching a flock of drones weave throug...",From watching a flock of drones weave through ...,430
1,Alejandro R. Gómez,[Satellite remote sensing for climate change m...,"[Python, R, MATLAB, Google Earth Engine, ENVI,...",Civil and Environmental Engineering,University of Cambridge,United Kingdom,[Deep Learning‑Based Atmospheric Correction fo...,2,21,4,[When I first examined a series of Sentinel‑2 ...,Engineering & Technology,AI & Autonomous Systems,151,generated_sop_151.pdf,"[""When I first examined a series of Sentinel\u...",When I first examined a series of Sentinel‑2 i...,445
2,Aisha Khan,"[Autonomous navigation for aerial robotics, Mu...","[Python, C++, Robot Operating System (ROS), Te...",Electrical and Computer Engineering,Massachusetts Institute of Technology,United States,[Learning Safe Maneuvers for Quadrotor Swarms ...,2,27,3,[From the moment I programmed my first line‑of...,Engineering & Technology,AI & Autonomous Systems,152,generated_sop_152.pdf,"[""From the moment I programmed my first line\u...",From the moment I programmed my first line‑of‑...,453
3,Mateo Alvarez,[Satellite remote sensing for environmental mo...,"[Python (NumPy, Pandas, Scikit‑learn), R (sp, ...",Civil and Environmental Engineering,University of Cambridge,United Kingdom,[Integrating SAR and Optical Data for Flood Ma...,3,45,4,"[Growing up along the Pacific coast of Chile, ...",Engineering & Technology,AI & Autonomous Systems,153,generated_sop_153.pdf,"[""Growing up along the Pacific coast of Chile,...","Growing up along the Pacific coast of Chile, I...",443
4,Aria L. Chen,[Robust perception for autonomous aerial vehic...,"[Python, C++, ROS2, TensorFlow, PyTorch, JAX, ...",Electrical Engineering and Computer Science,Massachusetts Institute of Technology,United States,[Safe Multi‑Agent Reinforcement Learning for U...,3,27,4,[From the moment I piloted a quadcopter throug...,Engineering & Technology,AI & Autonomous Systems,154,generated_sop_154.pdf,"[""From the moment I piloted a quadcopter throu...",From the moment I piloted a quadcopter through...,479


In [8]:
import torch


def append_categorical_embeddings(
    prefinal_dataset, categorical_cols, categorical_encoder, device=None
):
    """Batches the 'cat_ids' from the prefinal_dataset, runs them through the

    categorical_encoder, and appends the resulting normalized embedding as
    'cat_emb' to each row.
    """
    if len(prefinal_dataset) == 0:
        return prefinal_dataset

    if device is None:
        # Automatically detect the device from the encoder parameters
        device = next(categorical_encoder.parameters()).device

    # 1. Stack all 'cat_ids' tensors into a single 2D tensor
    # Shape: (num_rows, num_columns)
    cat_ids_stacked = torch.stack(
        [item["cat_ids"] for item in prefinal_dataset]
    ).to(device)

    # 2. Build the dictionary expected by the encoder's forward pass
    categorical_inputs = {}
    for idx, col in enumerate(categorical_cols):
        # Slice the column dimension for all rows: Shape (num_rows,)
        categorical_inputs[col] = cat_ids_stacked[:, idx]

    # 3. Run the forward pass with gradients disabled
    categorical_encoder.eval()
    with torch.no_grad():
        # Shape: (num_rows, num_columns * 32)
        all_embeddings = categorical_encoder(categorical_inputs)

    # 4. Append each row's 1D embedding tensor to the dataset dicts
    for idx, item in enumerate(prefinal_dataset):
        # Shape: (num_columns * 32,)
        item["cat_emb"] = all_embeddings[idx]

    return prefinal_dataset


In [17]:
import sys
import os

# Use __file__ if running as a script, fallback to os.getcwd() in Jupyter notebook
try:
    notebook_dir = os.path.dirname(os.path.abspath(__file__))
except NameError:
    notebook_dir = os.getcwd()

# Go up two levels from 'src/utils' to reach the project root directory
project_root = os.path.abspath(os.path.join(notebook_dir, "..", ".."))

if project_root not in sys.path:
    sys.path.append(project_root)

from src.model_utils.query_tower_utils.categorical_encoder import QueryCategoricalEncoder
from src.model_utils.query_tower_utils.numerical_encoder import QueryNumercalEncoder
# from src.model_utils.query_tower_utils.numerical_encoder import QueryNumericalEncoder


In [12]:
# 1. Instantiate the encoder and move it to the device
# Note: The QueryCategoricalEncoder class in categorical_encoder.py handles embedding_dim=32 by default
categorical_encoder = QueryCategoricalEncoder(categorical_maps).to(device)

# 2. Append 'cat_emb' to your existing PHD_prefinal_dataset
PHD_prefinal_dataset = append_categorical_embeddings(
    prefinal_dataset=PHD_prefinal_dataset,
    categorical_cols=categorical_col, # e.g., ["department", "university", "country"]
    categorical_encoder=categorical_encoder,
    device=device
)

# 3. Verify the result
print("First row of the updated dataset:")
print(PHD_prefinal_dataset[0].keys())
print("cat_emb shape:", PHD_prefinal_dataset[0]["cat_emb"].shape)


First row of the updated dataset:
dict_keys(['scholar_id', 'cat_ids', 'cat_emb'])
cat_emb shape: torch.Size([160])


In [14]:
PHD_prefinal_dataset[0]

{'scholar_id': 150,
 'cat_ids': tensor([117,   8,   7,   1,   4]),
 'cat_emb': tensor([-0.0536, -0.1200, -0.0054,  0.0758, -0.0147,  0.0565, -0.0610, -0.0036,
         -0.0930,  0.0555, -0.0396,  0.0412, -0.0317, -0.0834,  0.0974, -0.1482,
         -0.0185,  0.0415, -0.1815, -0.1261,  0.0252, -0.0825, -0.0374,  0.0390,
          0.0032, -0.0658,  0.0674,  0.0053, -0.0444,  0.0385, -0.0905, -0.0797,
          0.0449,  0.1074,  0.0917, -0.0215,  0.0486,  0.0277,  0.0064,  0.0375,
         -0.0416,  0.0087, -0.0865,  0.0190, -0.1329, -0.0959,  0.0765,  0.0646,
          0.0882,  0.0114,  0.1502, -0.0247, -0.0753, -0.0546, -0.0567, -0.0513,
          0.0158, -0.0493,  0.1088, -0.0347,  0.0529,  0.0226,  0.0123,  0.0998,
          0.1051,  0.1086,  0.0592,  0.0393, -0.0944,  0.0304, -0.0555,  0.1885,
         -0.0316,  0.0072,  0.0061, -0.0697,  0.0126, -0.0754, -0.0975,  0.0199,
         -0.0057, -0.0091, -0.0711, -0.0760, -0.0594, -0.0393,  0.1275, -0.0041,
          0.0191, -0.0218,  0.0

In [19]:
from sklearn.preprocessing import StandardScaler
import pandas as pd
import torch


def append_numerical_features(
    prefinal_dataset, df, numerical_cols, numerical_encoder, scaler=None, device=None
):
    """Cleans numerical columns (converting 'UNK' or other strings to 0), scales

    them, runs them through the numerical encoder, and appends 'num_ids' (scaled
    values) and 'num_emb' (numerical embeddings) to each row in the dataset.
    """
    if len(prefinal_dataset) == 0:
        return prefinal_dataset, scaler

    if device is None:
        device = next(numerical_encoder.parameters()).device

    # 1. Clean the numerical columns: convert strings like 'UNK' to NaN and fill with 0
    clean_df = pd.DataFrame(index=df.index)
    for col in numerical_cols:
        clean_df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0.0)

    # 2. Scale the cleaned numerical columns
    if scaler is None:
        scaler = StandardScaler()
        numeric_values = scaler.fit_transform(clean_df[numerical_cols])
    else:
        numeric_values = scaler.transform(clean_df[numerical_cols])

    # Convert to PyTorch float32 tensor
    numeric_tensor = torch.tensor(
        numeric_values, dtype=torch.float32, device=device
    )

    # 3. Run the numerical encoder
    numerical_encoder.eval()
    with torch.no_grad():
        numerical_embeddings = numerical_encoder(numeric_tensor)

    # 4. Append 'num_ids' and 'num_emb' to each row
    for idx, item in enumerate(prefinal_dataset):
        item["num_ids"] = numeric_tensor[idx]
        item["num_emb"] = numerical_embeddings[idx]

    return prefinal_dataset, scaler


In [20]:
numerical_col = ["publication_count", "years_experience", "citation_count"]
# 1. Instantiate the numerical encoder model and move to device
numerical_encoder = QueryNumercalEncoder(
    input_dim=len(numerical_col), 
    output_dim=32
).to(device)

# 2. Append the numerical values and embeddings
PHD_prefinal_dataset, fitted_scaler = append_numerical_features(
    prefinal_dataset=PHD_prefinal_dataset,
    df=df,
    numerical_cols=numerical_col, # ["publication_count", "years_experience", "citation_count"]
    numerical_encoder=numerical_encoder,
    scaler=None, # Will fit a new StandardScaler internally
    device=device
)

# 3. Verify the result
print("First row keys in the updated dataset:")
print(PHD_prefinal_dataset[0].keys())
print("num_ids (scaled inputs):", PHD_prefinal_dataset[0]["num_ids"])
print("num_emb (embedding):", PHD_prefinal_dataset[0]["num_emb"])


First row keys in the updated dataset:
dict_keys(['scholar_id', 'cat_ids', 'cat_emb', 'num_ids', 'num_emb'])
num_ids (scaled inputs): tensor([-0.8663, -1.2662,  0.4375])
num_emb (embedding): tensor([ 0.0185,  0.1123,  0.1711, -0.0622,  0.0701,  0.0703, -0.2171,  0.1416,
         0.3679,  0.1457, -0.3624,  0.1867,  0.0149,  0.1227, -0.0679,  0.0362,
        -0.2328, -0.1812,  0.0609, -0.0444,  0.0026, -0.1331, -0.1369, -0.2902,
        -0.2388,  0.2628,  0.0640, -0.3037,  0.1057, -0.2122, -0.0199,  0.2011])


In [21]:
PHD_prefinal_dataset[0]

{'scholar_id': 150,
 'cat_ids': tensor([117,   8,   7,   1,   4]),
 'cat_emb': tensor([-0.0536, -0.1200, -0.0054,  0.0758, -0.0147,  0.0565, -0.0610, -0.0036,
         -0.0930,  0.0555, -0.0396,  0.0412, -0.0317, -0.0834,  0.0974, -0.1482,
         -0.0185,  0.0415, -0.1815, -0.1261,  0.0252, -0.0825, -0.0374,  0.0390,
          0.0032, -0.0658,  0.0674,  0.0053, -0.0444,  0.0385, -0.0905, -0.0797,
          0.0449,  0.1074,  0.0917, -0.0215,  0.0486,  0.0277,  0.0064,  0.0375,
         -0.0416,  0.0087, -0.0865,  0.0190, -0.1329, -0.0959,  0.0765,  0.0646,
          0.0882,  0.0114,  0.1502, -0.0247, -0.0753, -0.0546, -0.0567, -0.0513,
          0.0158, -0.0493,  0.1088, -0.0347,  0.0529,  0.0226,  0.0123,  0.0998,
          0.1051,  0.1086,  0.0592,  0.0393, -0.0944,  0.0304, -0.0555,  0.1885,
         -0.0316,  0.0072,  0.0061, -0.0697,  0.0126, -0.0754, -0.0975,  0.0199,
         -0.0057, -0.0091, -0.0711, -0.0760, -0.0594, -0.0393,  0.1275, -0.0041,
          0.0191, -0.0218,  0.0

In [ ]:
import json
import torch


def convert_tensors_to_lists(obj):
    """Recursively converts PyTorch tensors in a dictionary/list to standard

    Python lists so they can be serialized to JSON.
    """
    if isinstance(obj, dict):
        return {k: convert_tensors_to_lists(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [convert_tensors_to_lists(item) for item in obj]
    elif isinstance(obj, torch.Tensor):
        # Move tensor to CPU, detach from graph, and convert to list
        return obj.detach().cpu().tolist()
    else:
        return obj


# 1. Ensure the directory exists
os.makedirs(
    r"C:\Users\ps302\OneDrive\Desktop\Recommend\src\data\processed\prefinal_dataset",
    exist_ok=True,
)

# 2. Convert all tensors in the dataset to standard lists
serializable_dataset = convert_tensors_to_lists(PHD_prefinal_dataset)

# 3. Save the JSON file in text write mode ("w") instead of binary write mode ("wb")
file_path = r"C:\Users\ps302\OneDrive\Desktop\Recommend\src\data\processed\prefinal_dataset\PHD_prefinal_dataset.json"
with open(file_path, "w") as file:
    json.dump(serializable_dataset, file, indent=4)

print(f"Saved successfully to: {file_path}")


Saved successfully to: C:\Users\ps302\OneDrive\Desktop\Recommend\src\data\processed\prefinal_dataset\PHD_prefinal_dataset.json


In [27]:
import pandas as pd
import torch


def convert_tensors_to_lists(obj):
    """Recursively converts PyTorch tensors to standard Python lists."""
    if isinstance(obj, dict):
        return {k: convert_tensors_to_lists(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [convert_tensors_to_lists(item) for item in obj]
    elif isinstance(obj, torch.Tensor):
        return obj.detach().cpu().tolist()
    else:
        return obj


# 1. Convert any PyTorch tensors in the dataset to standard lists
clean_dataset = convert_tensors_to_lists(PHD_prefinal_dataset)

# 2. Create a pandas DataFrame
df_csv = pd.DataFrame(clean_dataset)

# 3. Save to CSV
csv_path = r"C:\Users\ps302\OneDrive\Desktop\Recommend\src\data\processed\prefinal_dataset\PHD_prefinal_dataset.csv"
df_csv.to_csv(csv_path, index=False)

print(f"Saved dataset as CSV successfully at:\n{csv_path}")


Saved dataset as CSV successfully at:
C:\Users\ps302\OneDrive\Desktop\Recommend\src\data\processed\prefinal_dataset\PHD_prefinal_dataset.csv


In [ ]:
text_col = ["research_interests", "publications", "full_sop_markdown", "expertise"]

In [22]:
text_df = df[["research_interests", "publications", "full_sop_markdown", "expertise"]]
text_df.head()

,research_interests,publications,full_sop_markdown,expertise
0,"[Autonomous navigation for aerial robotics, Mu...",[Learning Safe Maneuvers for Quadrotor Swarms ...,From watching a flock of drones weave through ...,"[Python, C++, Robot Operating System (ROS), Te..."
1,[Satellite remote sensing for climate change m...,[Deep Learning‑Based Atmospheric Correction fo...,When I first examined a series of Sentinel‑2 i...,"[Python, R, MATLAB, Google Earth Engine, ENVI,..."
2,"[Autonomous navigation for aerial robotics, Mu...",[Learning Safe Maneuvers for Quadrotor Swarms ...,From the moment I programmed my first line‑of‑...,"[Python, C++, Robot Operating System (ROS), Te..."
3,[Satellite remote sensing for environmental mo...,[Integrating SAR and Optical Data for Flood Ma...,"Growing up along the Pacific coast of Chile, I...","[Python (NumPy, Pandas, Scikit‑learn), R (sp, ..."
4,[Robust perception for autonomous aerial vehic...,[Safe Multi‑Agent Reinforcement Learning for U...,From the moment I piloted a quadcopter through...,"[Python, C++, ROS2, TensorFlow, PyTorch, JAX, ..."


In [28]:
import pandas as pd


def generate_combined_texts(df, text_cols):
    """Combines specified text columns for each row into a single string.

    Automatically handles lists by joining them with commas and prefixes each
    field with its formatted column name.
    """
    combined_texts = []

    for _, row in df.iterrows():
        row_pieces = []
        for col in text_cols:
            val = row[col]

            # Skip null values
            if pd.isna(val) if not isinstance(val, list) else False:
                continue

            # If the value is a list, join elements with commas
            if isinstance(val, list):
                val_str = ", ".join(
                    [str(item).strip() for item in val if pd.notna(item)]
                )
            else:
                val_str = str(val).strip()

            # Format as "Column name: text value"
            if val_str:
                col_prefix = col.replace("_", " ").capitalize()
                row_pieces.append(f"{col_prefix}: {val_str}")

        # Combine all parts with a period and space
        row_text = ". ".join(row_pieces)
        if row_text:
            row_text += "."

        combined_texts.append(row_text)

    return combined_texts


In [29]:
# 1. Define the text columns you want to merge
text_col = ["research_interests", "publications", "full_sop_markdown"]

# 2. Initialize your target DataFrame if you haven't already
text_df = pd.DataFrame()

# 3. Generate the single combined string for each row and save to text_df["texts"]
text_df["texts"] = generate_combined_texts(df, text_col)

# 4. View a sample of the generated texts
print(text_df["texts"].iloc[0][:500])  # Prints first 500 characters of row 0


Research interests: Autonomous navigation for aerial robotics, Multi‑agent reinforcement learning, Safety‑critical AI for UAV swarms, Sensor fusion and visual‑inertial odometry, Edge AI and low‑power inference, Explainable decision‑making in autonomous systems, Human‑robot interaction in mixed‑initiative missions. Publications: Learning Safe Maneuvers for Quadrotor Swarms via Constrained Reinforcement Learning, Edge‑Optimized Visual‑Inertial Odometry for Low‑Power UAVs. Full sop markdown: From w


In [31]:
text_df.at[0,"texts"]

'Research interests: Autonomous navigation for aerial robotics, Multi‑agent reinforcement learning, Safety‑critical AI for UAV swarms, Sensor fusion and visual‑inertial odometry, Edge AI and low‑power inference, Explainable decision‑making in autonomous systems, Human‑robot interaction in mixed‑initiative missions. Publications: Learning Safe Maneuvers for Quadrotor Swarms via Constrained Reinforcement Learning, Edge‑Optimized Visual‑Inertial Odometry for Low‑Power UAVs. Full sop markdown: From watching a flock of drones weave through a canyon during a summer research retreat, I realized that autonomous systems must reconcile agility with provable safety. This observation sparked my fascination with the paradox of learning‑based autonomy: how to endow machines with the flexibility of deep reinforcement learning while guaranteeing that their actions never compromise mission‑critical constraints.\n\nMy central research question is: How can safety‑critical guarantees be integrated into le

In [34]:
text_df.sample(2)

,texts
761,Research interests: Protein structure predicti...
736,Research interests: Remote sensing for nitroge...


In [36]:
import torch


def append_text_embeddings_raw(
    prefinal_dataset, texts, sentence_transformer, device=None
):
    """Encodes a list of text descriptions using a raw SentenceTransformer

    model, and appends each resulting embedding as 'text_emb' to the
    corresponding row in prefinal_dataset.
    """
    if len(prefinal_dataset) == 0:
        return prefinal_dataset

    if device is None:
        # Get the device from the sentence transformer parameters
        device = next(sentence_transformer.parameters()).device

    # Ensure texts is a Python list of strings
    if hasattr(texts, "tolist"):
        text_list = texts.tolist()
    else:
        text_list = list(texts)

    # Generate embeddings using the raw SentenceTransformer model
    with torch.no_grad():
        all_embeddings = sentence_transformer.encode(
            text_list,
            convert_to_tensor=True,  # Return PyTorch tensors directly
            normalize_embeddings=True,  # L2-normalize the output embeddings
            show_progress_bar=True,
            device=device,
        )

    # Append 'text_emb' to each row in the dataset
    for idx, item in enumerate(prefinal_dataset):
        # Shape will be (384,) for all-MiniLM-L6-v2
        item["text_emb"] = all_embeddings[idx]

    return prefinal_dataset


In [37]:
from sentence_transformers import SentenceTransformer

# 1. Initialize raw SentenceTransformer and move to device
text_encoder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
text_encoder.to(device)

# 2. Append embeddings directly using the raw model
PHD_prefinal_dataset = append_text_embeddings_raw(
    prefinal_dataset=PHD_prefinal_dataset,
    texts=text_df["texts"],
    sentence_transformer=text_encoder,
    device=device
)

# 3. Verify the result
print("First row keys in the updated dataset:")
print(PHD_prefinal_dataset[0].keys())
print("text_emb shape (MiniLM output dimension):", PHD_prefinal_dataset[0]["text_emb"].shape)


c:\Users\ps302\anaconda3\envs\genai\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Batches: 100%|██████████| 27/27 [00:26<00:00,  1.01it/s]

First row keys in the updated dataset:
dict_keys(['scholar_id', 'cat_ids', 'cat_emb', 'num_ids', 'num_emb', 'text_emb'])
text_emb shape (MiniLM output dimension): torch.Size([384])


In [38]:
PHD_prefinal_dataset[0]

{'scholar_id': 150,
 'cat_ids': tensor([117,   8,   7,   1,   4]),
 'cat_emb': tensor([-0.0536, -0.1200, -0.0054,  0.0758, -0.0147,  0.0565, -0.0610, -0.0036,
         -0.0930,  0.0555, -0.0396,  0.0412, -0.0317, -0.0834,  0.0974, -0.1482,
         -0.0185,  0.0415, -0.1815, -0.1261,  0.0252, -0.0825, -0.0374,  0.0390,
          0.0032, -0.0658,  0.0674,  0.0053, -0.0444,  0.0385, -0.0905, -0.0797,
          0.0449,  0.1074,  0.0917, -0.0215,  0.0486,  0.0277,  0.0064,  0.0375,
         -0.0416,  0.0087, -0.0865,  0.0190, -0.1329, -0.0959,  0.0765,  0.0646,
          0.0882,  0.0114,  0.1502, -0.0247, -0.0753, -0.0546, -0.0567, -0.0513,
          0.0158, -0.0493,  0.1088, -0.0347,  0.0529,  0.0226,  0.0123,  0.0998,
          0.1051,  0.1086,  0.0592,  0.0393, -0.0944,  0.0304, -0.0555,  0.1885,
         -0.0316,  0.0072,  0.0061, -0.0697,  0.0126, -0.0754, -0.0975,  0.0199,
         -0.0057, -0.0091, -0.0711, -0.0760, -0.0594, -0.0393,  0.1275, -0.0041,
          0.0191, -0.0218,  0.0

In [39]:
import pandas as pd
import torch


def convert_tensors_to_lists(obj):
    """Recursively converts PyTorch tensors to standard Python lists."""
    if isinstance(obj, dict):
        return {k: convert_tensors_to_lists(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [convert_tensors_to_lists(item) for item in obj]
    elif isinstance(obj, torch.Tensor):
        return obj.detach().cpu().tolist()
    else:
        return obj


# 1. Convert any PyTorch tensors in the dataset to standard lists
clean_dataset = convert_tensors_to_lists(PHD_prefinal_dataset)

# 2. Create a pandas DataFrame
df_csv = pd.DataFrame(clean_dataset)

# 3. Save to CSV
csv_path = r"C:\Users\ps302\OneDrive\Desktop\Recommend\src\data\processed\prefinal_dataset\PHD_prefinal_dataset.csv"
df_csv.to_csv(csv_path, index=False)

print(f"Saved dataset as CSV successfully at:\n{csv_path}")


Saved dataset as CSV successfully at:
C:\Users\ps302\OneDrive\Desktop\Recommend\src\data\processed\prefinal_dataset\PHD_prefinal_dataset.csv


In [40]:
import json
import torch


def convert_tensors_to_lists(obj):
    """Recursively converts PyTorch tensors in a dictionary/list to standard

    Python lists so they can be serialized to JSON.
    """
    if isinstance(obj, dict):
        return {k: convert_tensors_to_lists(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [convert_tensors_to_lists(item) for item in obj]
    elif isinstance(obj, torch.Tensor):
        # Move tensor to CPU, detach from graph, and convert to list
        return obj.detach().cpu().tolist()
    else:
        return obj


# 1. Ensure the directory exists
os.makedirs(
    r"C:\Users\ps302\OneDrive\Desktop\Recommend\src\data\processed\prefinal_dataset",
    exist_ok=True,
)

# 2. Convert all tensors in the dataset to standard lists
serializable_dataset = convert_tensors_to_lists(PHD_prefinal_dataset)

# 3. Save the JSON file in text write mode ("w") instead of binary write mode ("wb")
file_path = r"C:\Users\ps302\OneDrive\Desktop\Recommend\src\data\processed\prefinal_dataset\PHD_prefinal_dataset.json"
with open(file_path, "w") as file:
    json.dump(serializable_dataset, file, indent=4)

print(f"Saved successfully to: {file_path}")


Saved successfully to: C:\Users\ps302\OneDrive\Desktop\Recommend\src\data\processed\prefinal_dataset\PHD_prefinal_dataset.json


In [41]:
# Save the dataset directly (keeps tensors, devices, and data types intact)
torch.save(
    PHD_prefinal_dataset,
    r"C:\Users\ps302\OneDrive\Desktop\Recommend\src\data\processed\prefinal_dataset\PHD_prefinal_dataset.pt"
)


In [ ]:
# # Load the dataset back with tensors intact
# PHD_prefinal_dataset = torch.load(
#     r"C:\Users\ps302\OneDrive\Desktop\Recommend\src\data\processed\prefinal_dataset\PHD_prefinal_dataset.pt"
# )
